# Mimi Neural Audio Codec — Hands-on Test

Test Kyutai's Mimi codec: encode audio to tokens, decode back, and analyze codebook behavior.

**Runtime**: GPU (T4 is fine) — `Runtime > Change runtime type > T4 GPU`

## 1. Install Dependencies

In [ ]:
!pip install -q transformers torch torchaudio matplotlib numpy IPython librosa

## 2. Load Mimi Model

In [ ]:
import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display, HTML
from transformers import MimiModel, AutoFeatureExtractor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load Mimi
model = MimiModel.from_pretrained("kyutai/mimi").to(device)
feature_extractor = AutoFeatureExtractor.from_pretrained("kyutai/mimi")

print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
print(f"Sample rate: {feature_extractor.sampling_rate} Hz")
print(f"Codebooks: {model.config.num_quantizers}")
print(f"Codebook size: {model.config.codebook_size}")

## 3. Create Test Audio

We'll generate a simple sine wave first, then test with real speech.

In [ ]:
SAMPLE_RATE = 24000

def make_sine(freq_hz, duration_s=2.0, sr=SAMPLE_RATE):
    """Generate a sine wave."""
    t = torch.linspace(0, duration_s, int(sr * duration_s))
    return torch.sin(2 * np.pi * freq_hz * t).unsqueeze(0)  # [1, samples]

def play(audio, sr=SAMPLE_RATE, label=""):
    """Display audio player with label."""
    if label:
        display(HTML(f"<b>{label}</b>"))
    if isinstance(audio, torch.Tensor):
        audio = audio.detach().cpu().numpy()
    if audio.ndim > 1:
        audio = audio.squeeze()
    display(Audio(audio, rate=sr))

# Test with a 440 Hz sine wave
sine_440 = make_sine(440, duration_s=2.0)
play(sine_440, label="Original: 440 Hz sine wave")

## 4. Encode and Decode

Core Mimi workflow: Audio -> Tokens -> Audio

In [ ]:
@torch.no_grad()
def encode(audio, num_codebooks=None):
    """Encode audio to Mimi tokens.
    
    Args:
        audio: [1, samples] tensor at 24kHz
        num_codebooks: how many codebooks to use (1-8, default all)
    Returns:
        codes: [1, num_codebooks, num_frames] tensor of token IDs
    """
    inputs = feature_extractor(
        raw_audio=audio.squeeze().numpy(),
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt"
    ).to(device)
    
    output = model.encode(
        inputs["input_values"],
        num_quantizers=num_codebooks
    )
    return output.audio_codes  # [1, num_codebooks, num_frames]


@torch.no_grad()
def decode(codes):
    """Decode Mimi tokens back to audio.
    
    Returns:
        audio: [1, 1, samples] tensor at 24kHz
    """
    output = model.decode(codes)
    return output.audio_values  # [1, 1, samples]


# Encode the sine wave
codes = encode(sine_440)
print(f"Input:  {sine_440.shape[1]} samples ({sine_440.shape[1]/SAMPLE_RATE:.2f}s)")
print(f"Codes:  {codes.shape}  (batch, codebooks, frames)")
print(f"Frame rate: {codes.shape[2] / (sine_440.shape[1]/SAMPLE_RATE):.1f} Hz")
print(f"Bitrate: {codes.shape[1] * codes.shape[2] * 11 / (sine_440.shape[1]/SAMPLE_RATE) / 1000:.2f} kbps")
print(f"\nCodebook 1 (semantic) first 10 tokens: {codes[0, 0, :10].tolist()}")
print(f"Codebook 2 (acoustic) first 10 tokens: {codes[0, 1, :10].tolist()}")

# Decode back
reconstructed = decode(codes)
print(f"\nReconstructed: {reconstructed.shape[2]} samples")
play(reconstructed.squeeze(0), label="Reconstructed: 440 Hz sine (all 8 codebooks)")

## 5. Codebook Ablation — What Does Each Codebook Capture?

Decode with 1, 2, 3, ... 8 codebooks to hear the progressive quality improvement.

In [ ]:
# Download a real speech sample from LibriSpeech (Open Speech dataset)
import urllib.request
import os

SPEECH_URL = "https://www.openslr.org/resources/12/test-clean-wav/1089-134686-0000.wav"
SPEECH_FILE = "/tmp/test_speech.wav"

# Try LibriSpeech first, fall back to generating speech with gTTS
if not os.path.exists(SPEECH_FILE):
    try:
        urllib.request.urlretrieve(SPEECH_URL, SPEECH_FILE)
        print("Downloaded LibriSpeech sample.")
    except Exception as e:
        print(f"Download failed ({e}), generating speech with gTTS instead...")
        try:
            from gtts import gTTS
        except ImportError:
            import subprocess
            subprocess.check_call(["pip", "install", "-q", "gtts"])
            from gtts import gTTS
        import io
        tts = gTTS(text="The quick brown fox jumps over the lazy dog. This is a test of the Mimi neural audio codec.", lang='en')
        tts.save(SPEECH_FILE)
        print("Generated speech with gTTS.")

speech, sr = torchaudio.load(SPEECH_FILE)
if sr != SAMPLE_RATE:
    speech = torchaudio.functional.resample(speech, sr, SAMPLE_RATE)

# Take first 5 seconds, mono
speech = speech[0:1, :SAMPLE_RATE * 5]
print(f"Speech: {speech.shape[1]} samples ({speech.shape[1]/SAMPLE_RATE:.1f}s)")
play(speech, label="Original speech")

In [ ]:
# Encode with all 8 codebooks
all_codes = encode(speech, num_codebooks=8)
print(f"Codes shape: {all_codes.shape}")

# Decode with 1, 2, 3, ... 8 codebooks
print("\nProgressive reconstruction:")
for n in range(1, 9):
    partial_codes = all_codes[:, :n, :]  # first n codebooks
    # Pad remaining codebooks with zeros
    padding = torch.zeros(
        1, 8 - n, all_codes.shape[2],
        dtype=all_codes.dtype, device=all_codes.device
    )
    padded_codes = torch.cat([partial_codes, padding], dim=1)
    
    recon = decode(padded_codes)
    label = f"Codebooks 1-{n}"
    if n == 1:
        label += " (semantic only — words/phonemes)"
    elif n == 2:
        label += " (+ coarse acoustic)"
    elif n == 4:
        label += " (+ speaker identity)"
    elif n == 8:
        label += " (full quality)"
    play(recon.squeeze(0), label=label)

## 6. Visualize Token Distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle("Token Distribution per Codebook", fontsize=14)

for i in range(8):
    ax = axes[i // 4][i % 4]
    tokens = all_codes[0, i, :].cpu().numpy()
    ax.hist(tokens, bins=50, color=f"C{i}", alpha=0.7)
    ax.set_title(f"CB {i+1} {'(semantic)' if i == 0 else '(acoustic)'}")
    ax.set_xlabel("Token ID")
    ax.set_ylabel("Count")
    unique = len(set(tokens.tolist()))
    ax.text(0.95, 0.95, f"{unique} unique", transform=ax.transAxes,
            ha='right', va='top', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize tokens as a heatmap over time
fig, ax = plt.subplots(figsize=(16, 4))
tokens_np = all_codes[0].cpu().numpy()  # [8, T]
im = ax.imshow(tokens_np, aspect='auto', cmap='viridis', interpolation='nearest')
ax.set_xlabel("Frame (12.5 Hz)")
ax.set_ylabel("Codebook")
ax.set_yticks(range(8))
ax.set_yticklabels([f"CB{i+1} {'(sem)' if i==0 else '(aco)'}" for i in range(8)])
ax.set_title("Mimi Token Heatmap")
plt.colorbar(im, label="Token ID")
plt.tight_layout()
plt.show()

## 7. Upload Your Own Audio

Upload a WAV/MP3 file to test Mimi on your own audio (e.g., Vietnamese speech).

In [ ]:
from google.colab import files

print("Upload a WAV or MP3 file:")
uploaded = files.upload()

for filename in uploaded:
    audio, sr = torchaudio.load(filename)
    if sr != SAMPLE_RATE:
        audio = torchaudio.functional.resample(audio, sr, SAMPLE_RATE)
    audio = audio[0:1]  # mono
    
    # Limit to 10 seconds
    max_samples = SAMPLE_RATE * 10
    if audio.shape[1] > max_samples:
        audio = audio[:, :max_samples]
    
    print(f"\n--- {filename} ---")
    print(f"Duration: {audio.shape[1]/SAMPLE_RATE:.2f}s")
    play(audio, label=f"Original: {filename}")
    
    codes = encode(audio, num_codebooks=8)
    print(f"Codes: {codes.shape}")
    
    recon = decode(codes)
    play(recon.squeeze(0), label=f"Reconstructed: {filename}")

## 8. Vietnamese Tone Test

Test if Mimi's semantic codebook (CB1) can distinguish Vietnamese tones.

**How to use**: Upload 6 audio files of the Vietnamese minimal tone pair:
- `ma.wav` (ma - ngang/level)
- `mà.wav` (mà - huyền/falling)  
- `má.wav` (má - sắc/rising)
- `mả.wav` (mả - hỏi/dipping)
- `mã.wav` (mã - ngã/broken)
- `mạ.wav` (mạ - nặng/heavy)

Or use Google TTS to generate them automatically below.

In [ ]:
# Option A: Generate Vietnamese tone pairs using gTTS
!pip install -q gtts

from gtts import gTTS
import io

tone_words = {
    "ma":  "ngang (level) — ghost",
    "mà":  "huyền (falling) — but",
    "má":  "sắc (rising) — mother",
    "mả":  "hỏi (dipping) — grave",
    "mã":  "ngã (broken) — horse/code",
    "mạ":  "nặng (heavy) — rice seedling",
}

tone_results = {}

for word, desc in tone_words.items():
    # Generate with gTTS
    tts = gTTS(text=word, lang='vi')
    buf = io.BytesIO()
    tts.write_to_fp(buf)
    buf.seek(0)
    
    # Save temp file and load with torchaudio
    tmp_path = f"/tmp/tone_{word}.mp3"
    with open(tmp_path, 'wb') as f:
        f.write(buf.read())
    
    audio, sr = torchaudio.load(tmp_path)
    if sr != SAMPLE_RATE:
        audio = torchaudio.functional.resample(audio, sr, SAMPLE_RATE)
    audio = audio[0:1]  # mono
    
    # Encode
    codes = encode(audio, num_codebooks=8)
    
    tone_results[word] = {
        "desc": desc,
        "audio": audio,
        "codes": codes,
        "semantic_tokens": codes[0, 0, :].cpu().tolist(),
        "acoustic_tokens": codes[0, 1, :].cpu().tolist(),
    }
    
    print(f"{word} ({desc})")
    print(f"  Semantic (CB1): {codes[0, 0, :].cpu().tolist()}")
    print(f"  Acoustic (CB2): {codes[0, 1, :].cpu().tolist()}")
    play(audio, label=f"{word} — {desc}")
    print()

In [ ]:
# Analyze: Are semantic tokens different across tones?
print("=" * 60)
print("ANALYSIS: Can Mimi's semantic codebook distinguish tones?")
print("=" * 60)

words = list(tone_results.keys())

# Compare semantic token overlap between all pairs
print("\nSemantic token (CB1) overlap matrix:")
print(f"{'':>6}", end="")
for w in words:
    print(f"{w:>8}", end="")
print()

for w1 in words:
    print(f"{w1:>6}", end="")
    t1 = set(tone_results[w1]["semantic_tokens"])
    for w2 in words:
        t2 = set(tone_results[w2]["semantic_tokens"])
        if len(t1 | t2) > 0:
            overlap = len(t1 & t2) / len(t1 | t2) * 100
        else:
            overlap = 0
        print(f"{overlap:>7.1f}%", end="")
    print()

print("\n--- Interpretation ---")
print("High overlap (>80%) = Mimi CANNOT distinguish these tones in CB1")
print("Low overlap  (<40%) = Mimi CAN distinguish these tones in CB1")
print("\nIf all tones have high overlap in CB1, tonal info is in CB2-8 (acoustic),")
print("which means fine-tuning Mimi's semantic codebook for Vietnamese is needed.")

In [ ]:
# Visualize semantic tokens for each tone
fig, axes = plt.subplots(len(words), 1, figsize=(14, len(words) * 1.5), sharex=True)
fig.suptitle("Semantic Tokens (CB1) for Vietnamese Tones of 'ma'", fontsize=13)

for i, word in enumerate(words):
    tokens = tone_results[word]["semantic_tokens"]
    axes[i].bar(range(len(tokens)), tokens, color=f"C{i}", alpha=0.8)
    axes[i].set_ylabel(f"{word}")
    axes[i].set_ylim(0, 2048)
    desc_short = tone_results[word]["desc"].split("—")[0].strip()
    axes[i].text(0.98, 0.85, desc_short, transform=axes[i].transAxes,
                 ha='right', va='top', fontsize=9, style='italic')

axes[-1].set_xlabel("Frame")
plt.tight_layout()
plt.show()

## 9. Reconstruction Quality Metrics

In [ ]:
def compute_metrics(original, reconstructed):
    """Compute basic audio quality metrics."""
    orig = original.squeeze().cpu().numpy()
    recon = reconstructed.squeeze().cpu().numpy()
    
    # Match lengths
    min_len = min(len(orig), len(recon))
    orig = orig[:min_len]
    recon = recon[:min_len]
    
    # SNR
    noise = orig - recon
    snr = 10 * np.log10(np.sum(orig**2) / (np.sum(noise**2) + 1e-10))
    
    # Correlation
    corr = np.corrcoef(orig, recon)[0, 1]
    
    # L1 error
    l1 = np.mean(np.abs(noise))
    
    return {"SNR (dB)": snr, "Correlation": corr, "L1 Error": l1}


# Test with speech
print("Reconstruction quality by number of codebooks:")
print(f"{'Codebooks':>10} {'SNR (dB)':>10} {'Correlation':>12} {'L1 Error':>10}")
print("-" * 45)

speech_codes = encode(speech, num_codebooks=8)

for n in range(1, 9):
    partial = speech_codes[:, :n, :]
    padding = torch.zeros(
        1, 8 - n, speech_codes.shape[2],
        dtype=speech_codes.dtype, device=speech_codes.device
    )
    padded = torch.cat([partial, padding], dim=1)
    recon = decode(padded)
    
    m = compute_metrics(speech, recon)
    print(f"{n:>10} {m['SNR (dB)']:>10.2f} {m['Correlation']:>12.4f} {m['L1 Error']:>10.6f}")

## 10. Experiment: Swap Semantic Tokens Between Utterances

If we take CB1 (semantic) from utterance A and CB2-8 (acoustic) from utterance B,
we should get the **words of A** spoken in the **voice of B**.

In [ ]:
# This experiment requires two different speech samples
# We'll use the first and second halves of our speech sample as A and B
half = speech.shape[1] // 2
speech_a = speech[:, :half]
speech_b = speech[:, half:]

codes_a = encode(speech_a, num_codebooks=8)
codes_b = encode(speech_b, num_codebooks=8)

# Match frame lengths
min_frames = min(codes_a.shape[2], codes_b.shape[2])
codes_a = codes_a[:, :, :min_frames]
codes_b = codes_b[:, :, :min_frames]

play(speech_a, label="Speech A (first half)")
play(speech_b, label="Speech B (second half)")

# Swap: semantic from A + acoustic from B
swapped = torch.cat([
    codes_a[:, 0:1, :],   # CB1 (semantic) from A
    codes_b[:, 1:, :],    # CB2-8 (acoustic) from B
], dim=1)

recon_swapped = decode(swapped)
play(recon_swapped.squeeze(0), label="Swapped: semantic(A) + acoustic(B)")
print("\nIf Mimi's split works well, you should hear the WORDS from A")
print("spoken with acoustic characteristics closer to B.")

## Summary

What we tested:
1. **Encode/Decode** — audio -> 8 codebooks at 12.5 Hz -> audio
2. **Codebook ablation** — progressive quality from CB1 (robotic) to CB8 (full)
3. **Token distributions** — how codebook entries are used
4. **Vietnamese tones** — can CB1 distinguish tones?
5. **Reconstruction metrics** — SNR and correlation per codebook count
6. **Semantic/acoustic swap** — disentanglement test

### Key questions for your Vietnamese project:
- Does CB1 produce **different** tokens for different tones of the same syllable?
- If not, tonal information lives in CB2-8, and you'll need to fine-tune Mimi
- Upload real Vietnamese speech recordings for more meaningful results